# 01 Intro of Pydantic
Pydantic for data validation
in python there is no concept of static typing, its dynamic
so no direct wau to validate types

In [ ]:
def insert_into_database(name, aget):
    # Simulate database insertion
    print(f"Inserting {name}, {aget} into database")
insert_into_database("Alice", "twenty five")  # This should be an integer, but it's a string

# now above code will run without any error, but it will create issues later when we try to use the age as an integer
# to solve this we added type hints while defining the function

def insert_into_database(name: str, age: int):
    # Simulate database insertion
    print(f"Inserting {name}, {age} into database")
insert_into_database("Alice", -25)  # This is integer but negative, which is invalid for age

# so we make a custom validation function inside the function along with type validation
def insert_into_database(name: str, age: int):
    if type(name) == str and type(age) == int and age >= 0:
        # Simulate database insertion
        print(f"Inserting {name}, {age} into database")
    else:
        print("Invalid data provided")
insert_into_database("Alice", -25)  # This will now print "Invalid data provided

# but this approach is not scalable, what if we have multiple functions with multiple parameters
# also not suitable for production level code
# so we use pydantic library to handle all the validation for us

# Pydantic :
# 1. Define a Pydantic model (or can be say as class) that represents the ideal schema of the data.
#    - This includes the expected fields, their types, and any validation constraints (e.g., gt=0 for positive
# numbers).
# 2. Instantiate the model with raw input data (usually a dictionary or JSON-like structure).
#    - Pydantic will automatically validate the data and coerce it into the correct Python types (if
# possible).
#    - If the data doesn't meet the model's requirements, Pydantic raises a ValidationError.
# 3. Pass the validated model object to functions or use it throughout your codebase.
#    - This ensures that every part of your program works with clean, type-safe, and logically valid data.

Inserting Alice, twenty five into database
Inserting Alice, -25 into database
Invalid data provided


# 02 How Pydantic Works

In [ ]:
from pydantic import BaseModel
from typing import List, Dict

# Define a Pydantic model for our data (type hints + validation rules)
class Patient(BaseModel):
    name: str
    age: int
    # if you want to add more fields like weight, height etc you can simply add here

def insert_into_database(patient: Patient):
    # `patient` is the parameter name i.e local variable inside the function;
    # `: Patient` is a type hint (expects a Patient model).
    # Pydantic validates when you create `Patient(...)`, so this function can assume
    # the `patient` argument has valid `name` and `age` attributes.
    # Simulate database insertion
    print(f"Inserting {patient.name}, {patient.age} into database")

patient_info = {"name": "Alice", "age": 25}
# patient_info = {"name": "Alice", "age": "25"} # Pydantic will coerce this to int
# patient_info = {"name": "Alice", "age": "twenty five"} # This will raise a ValidationError

patient1 = Patient(**patient_info)
# since patient_info is dictionary we use ** to unpack it
# Pydantic will validate and create the model instance

insert_into_database(patient1)

Inserting Alice, 25 into database


now we will add more fields and type along with data validation

In [ ]:
from pydantic import BaseModel
from typing import List, Dict

class Patient(BaseModel):
    name: str
    age: int
    weight : float
    height : float
    married : bool
    allergies : List[str] # it should be a list and items inside list should be a string type
    contact_details : Dict[str, str] # it should be a dictionary and key:value of dictionary both should be string

def insert_into_database(patient: Patient):
    print(f"Inserting {patient.name}, {patient.age} into database")

patient_info = {
    "name": "Alice",
    "age": 25,
    "weight": 65.5,
    "height": 170.2,
    "married": True,
    "allergies": ["dust", "skin"],
    "contact_details": {"phone": "123-456-7890", "email": "alice@example.com"}
}
# if any datatype missmatched then function will not work

patient1 = Patient(**patient_info)
insert_into_database(patient1)

Inserting Alice, 25 into database


- now above if we are not passing all the fileds while creating object of class `Patient` then it will give an error
- however sometime we may need some fileds empty
- we can import `Optional` module from typing and use it to make field optional
- NOTE : while making field optional we need to set value as `None`
- We can also set default values for compulsory fileds

In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str
    age: int
    weight : float
    height : float
    married : bool = False # default value set to False
    allergies : Optional[List[str]] = None # this field is optional, value set to None if not provided
    contact_details : Dict[str, str]

def insert_into_database(patient: Patient):
    print(f"Inserting {patient.name}, {patient.age} into database")
    print(f"patient.allergies -->> {patient.allergies}")
    print(f"patient.married -->> {patient.married}")

patient_info = {
    "name": "Alice",
    "age": 25,
    "weight": 65.5,
    "height": 170.2,
    "married": True,
    # "allergies": ["dust", "skin"], # this field is optional
    "contact_details": {"phone": "123-456-7890", "email": "alice@example.com"}
}

patient1 = Patient(**patient_info)
insert_into_database(patient1)

Inserting Alice, 25 into database
patient.allergies -->> None
patient.married -->> True


- data / custom data validation using pydantic


In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str
    age: int
    email : EmailStr
    linkedIn : AnyUrl
    weight : float
    height : float
    married : bool = False
    allergies : Optional[List[str]] = None
    contact_details : Dict[str, str]

def insert_into_database(patient: Patient):
    print(f"Inserting {patient.name}, {patient.age} into database")
    print(f"patient.allergies -->> {patient.allergies}")
    print(f"patient.married -->> {patient.married}")
    print(f"patient.email -->> {patient.email}")
    print(f"patient.linkedIn -->> {patient.linkedIn}")

patient_info = {
    "name": "Alice",
    "age": 25,
    "email": "alice@example.com",
    "linkedIn": "https://www.linkedin.com/in/alice",
    "weight": 65.5,
    "height": 170.2,
    "married": True,
    # "allergies": ["dust", "skin"], # this field is optional
    "contact_details": {"phone": "123-456-7890", "address": "123 Main St"},
}

patient1 = Patient(**patient_info)
insert_into_database(patient1)

- `Field` function from `pydantic` is not just for validation but also for adding metadata and constraints to the model Fields and for this we have to use `Annotated` type from `typing` module

- actually pydantic is very smart enough just in case if weight is given as string it will convert it to float, but we can add extra validation using Field

weight : float = Field(gt=0)  -->> weight must be positive
istead of above line we should use below line
weight : float = Annotated[float, Field(gt=18, le=65 strict=True ,description="Weight of the patient in kilograms")]

In [ ]:
# !pip install 'pydantic[email]' # make sure to install this module

from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Full name of the patient less than 50 characters", examples=["John Doe", "Alice Smith"])]
    age: int = Field(ge=0, le=120)  # age must be between 0 and 120
    email : EmailStr
    linkedIn : AnyUrl
    # actually pydantic is very smart enough just in case if weight is given as string it will convert it to float, but we can add extra validation using Field
    # weight : float = Field(gt=0)  # weight must be positive
    weight : float = Annotated[float, Field(gt=18, le=65,strict=True,description="Weight of the patient in kilograms")]
    height : float
    married : Annotated[bool, Field(default=False, description="Marital status of the patient")] # first parameter is the datatype, second is the Field with default value and description
    allergies : Optional[List[str]] = None
    contact_details : Dict[str, str]

def insert_into_database(patient: Patient):
    print(f"Inserting {patient.name}, {patient.age} into database")
    print(f"patient.allergies -->> {patient.allergies}")
    print(f"patient.married -->> {patient.married}")
    print(f"patient.email -->> {patient.email}")
    print(f"patient.linkedIn -->> {patient.linkedIn}")

patient_info = {
    "name": "Alice",
    "age": 25,
    "email": "alice@example.com",
    "linkedIn": "https://www.linkedin.com/in/alice",
    "weight": 65.5,
    "height": 170.2,
    "married": True,
    # "allergies": ["dust", "skin"], # this field is optional
    "contact_details": {"phone": "123-456-7890", "address": "123 Main St"},
}

patient1 = Patient(**patient_info)
insert_into_database(patient1)

Inserting Alice, 25 into database
patient.allergies -->> None
patient.married -->> True
patient.email -->> alice@example.com
patient.linkedIn -->> https://www.linkedin.com/in/alice


# 03 Field Validator
- lest suppose we have a to build a model for a user extensively for corporate employees over their email address
with EmailStr type we can validate email address format
- but we also want to add a custom validation rule
- we want to make sure that the email address is valid and belongs to a specific domain

- this is one use case of field validator
- also just in case if you want a name to be always capitalized
- you can use field validator for that too

In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    # now lets consider we want to verify email from specific domain
    # for that we have to use decorator field_validator  as `@field_validator('email')`
    @field_validator('email')   # here we are specifying field name `email`
    @classmethod
    def email_validator(cls, value): # it will take cls (i.e class which is here as Patient) and value (here value of email) as parameters
        valid_domain = ['hdfc.com', 'icici.com'] # list of valid domains
        domain = value.split('@')[-1]  # extract domain from email
        if domain not in valid_domain:
            raise ValueError(f'Not a valid domain.')
        return value


    # another example of field validator to ensure name is capitalized
    @field_validator('name') # here we are specifying field name `name`
    @classmethod
    def name_must_be_capitalized(cls, value):
        return value.upper()  # return the capitalized value


def update_patient_data(patient: Patient):
    print(f"Updating patient data for {patient.name}")
    print(f"Email: {patient.email}")
    print(f"Age: {patient.age}")

patient_info = {
    "name": "Bob",
    "email": "bob@hdfc.com",
    # "email": "pranav@gmail.com",  # This will raise a ValidationError due to invalid domain
    "age": 40,
    "weight": 75.5,
    "married": True,
    "allergies": ["pollen", "nuts"],
    "contact_details": {"phone": "123-456-7890", "address": "123 Main St"}
}

patient1 = Patient(**patient_info)
update_patient_data(patient1)

- field validator is operated in two modes:
  1. mode = `after` (default): the validator is called after the value is parsed (after coercion) and validated by pydantic
  2. mode = `before`: the validator is called before the value is parsed and validated by pydantic

- type coercion means converting the input data type to the expected data type defined in the model.
- example: if age is provided as a string "25" but the model expects an int, pydantic will coerce it to int 25.

In [ ]:
from pydantic import BaseModel, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value): # value is 40 or "40" so if mode=`before` and value is "40" it will throw error but if mode=`after` it will be coerced to int 40
        if value > 0 and value <= 120:
            return value
        else:
            raise ValueError('Invalid age value')

def update_patient_data(patient: Patient):
    print(f"Updating patient data for {patient.name}")
    print(f"Age: {patient.age}")

patient_info = {
    "name": "Bob",
    "age": 40,
    # "age": "40",
    "weight": 75.5,
    "married": True,
    "allergies": ["pollen", "nuts"],
    "contact_details": {"phone": "123-456-7890", "address": "123 Main St"}
}

patient1 = Patient(**patient_info) # here validation and if needed type coercion happens where mode is `after` by default
update_patient_data(patient1)

Updating patient data for Bob
Age: 40


# 04 Model Valiadtor

- we save filed_validator on one particular field however what if we need to validate across multiple fields

**EXAMPLE:**
- we have class Patient with fields name, age, contact_details
- we want to ensure that if age < 18 and age > 60 then contact_details must include emergency_contact
- then only we can allow the model to be created

In [ ]:
from pydantic import BaseModel, model_validator
from typing import List, Dict

class Patient(BaseModel):
    name : str
    age : int
    contact_details : Dict[str, str]

    @model_validator(mode="after")
    def validate_emergency_contact(cls, model):
        if model.age < 18 or model.age > 60:
            if 'emergency_contact' not in model.contact_details:
                raise ValueError("emergency_contact is required for patients under 18 or over 60")
        return model


def validate_patient(patient: Patient):
    print("patient is valid")
    print(f"Name: {patient.name}, Age: {patient.age}")


patient_info = {"name": "Bob",
                "age": 65,
                "contact_details": {"phone": "123-456-7890", "emergency_contact": "123-456-7890"}
                }

patient1 = Patient(**patient_info)
validate_patient(patient1)

patient is valid
Name: Bob, Age: 65


/tmp/ipython-input-3781555886.py:9: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")


since getting warning as `PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")`


used self instead passing cls,model into validate_emergency_contact for **validator(mode="after")**

In [ ]:
from pydantic import BaseModel, model_validator
from typing import List, Dict

class Patient(BaseModel):
    name : str
    age : int
    contact_details : Dict[str, str]

    @model_validator(mode="after")
    def validate_emergency_contact(self):
        # Use an instance method for `mode='after'` validators (recommended since Pydantic v2.12).
        if self.age < 18 or self.age > 60:
            if 'emergency_contact' not in self.contact_details:
                raise ValueError("emergency_contact is required for patients under 18 or over 60")
        return self


def validate_patient(patient: Patient):
    print("patient is valid")
    print(f"Name: {patient.name}, Age: {patient.age}")


patient_info = {"name": "Bob",
                "age": 65,
                "contact_details": {"phone": "123-456-7890", "emergency_contact": "123-456-7890"}
                }

patient1 = Patient(**patient_info)
validate_patient(patient1)

patient is valid
Name: Bob, Age: 65


# 06 Computed Field
- its a field of a model where user does not provide value
- its value is computed using other fields of the model

**Example:**
- as we have a class Patient with fields name, age, height, weight
- now we want to compute BMI (Body Mass Index) based on height and weight which user is not aware of
- so we can define BMI as a computed field

In [ ]:
from pydantic import BaseModel, Field, computed_field
from typing import Optional, List, Dict

class Patient(BaseModel):
    name: str
    age: int
    height: float # in meters
    weight: float # in kilograms

    @computed_field
    @property
    def calculate_bmi(self) -> float:
        """Compute BMI using height and weight"""
        bmi = round(self.weight / (self.height ** 2), 2)
        return bmi

def all_info_of_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Height: {patient.height} meters")
    print(f"Patient Weight: {patient.weight} kilograms")
    print(f"Patient BMI: {patient.calculate_bmi}") # Accessing computed field which is here calculate_bmi, not bmi else keep both name same as bmi or calculate_bmi

patient_data = {
    "name": "John Doe",
    "age": 30,
    "height": 1.75,
    "weight": 70.0
    }

patient = Patient(**patient_data)
all_info_of_patient_data(patient)

Patient Name: John Doe
Patient Age: 30
Patient Height: 1.75 meters
Patient Weight: 70.0 kilograms
Patient BMI: 22.86


# 07 Nested Models
> use one Pydantic model inside another as a field type

- why to use nested models?
    - Better organization of related data (e.g., vitals, address, insurance)
    - Reusability: Use Vitals in multiple models (e.g., Patient, MedicalRecord)
    - Readability: Easier for developers and API consumers to understand
    - Validation: Nested models are validated automatically-no extra work needed

In [ ]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    state: str
    pincode: int

class Patient(BaseModel):
    name: str
    gender: str
    age: int
    # now we can specify address as str as address : str
    # and storing address as "house no 2, sec66, gurgaon, haryana,122002"
    # but if we want a pincode or city name
    # for this we have to create another model Address and use it here as a field type
    address: Address  # Nested Pydantic model

address_book = {
    "street": "house no 2, sec66",
    "city": "gurgaon",
    "state": "haryana",
    "pincode": 122002
    }
address1 = Address(**address_book)

patient_info = {
    "name": "Alice",
    "gender": "male",
    "age": 25,
    "address": address1 # passing Address model instance
    }
patient1 = Patient(**patient_info)


print(patient1)
print(patient1.address.city)  # Accessing nested model attribute   # it will print "gurgaon"
print(patient1.address.pincode)  # Accessing nested model attribute   # it will print "122002"

name='Alice' gender='male' age=25 address=Address(street='house no 2, sec66', city='gurgaon', state='haryana', pincode=122002)
gurgaon
122002


# 08 Serialzation and Deserialization
- Pydantic models can be easily serialized to and deserialized from various formats, such as Dictionary, JSON. This is particularly useful for web applications and APIs where data exchange is common.

In [ ]:
# We will use same previous code of Nested Models for demonstration
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    state: str
    pincode: int

class Patient(BaseModel):
    name: str
    gender: str
    age: int
    address: Address
    married: bool = False  # default value

address_book = {
    "street": "house no 2, sec66",
    "city": "gurgaon",
    "state": "haryana",
    "pincode": 122002
    }
address1 = Address(**address_book)

patient_info = {
    "name": "Alice",
    "gender": "male",
    "age": 25,
    "address": address1
    }
patient1 = Patient(**patient_info)


# Serialization: Convert Pydantic model to dictionary
dict_temp = patient1.model_dump()
print(dict_temp)
print(type(dict_temp))

# Serialization: Convert Pydantic model to JSON
json_temp = patient1.model_dump_json()
print(json_temp)
print(type(json_temp))

# visually you will not able to see much difference between outputs of model_dump and model_dump_json
# but if you check the type you will see model_dump gives dictionary and model_dump_json gives string


# you can control which fields to include or exclude during serialization
# For example, include only name, age in the dictionary representation
name_only_dict = patient1.model_dump(include={"name", "age"})
print(name_only_dict)

# For example, exclude age in the JSON representation
json_temp_exclude_age = patient1.model_dump_json(exclude={"age"})
print(json_temp_exclude_age)

# For example, include only street and city from the nested address in the dictionary representation
address_partial_dict = patient1.model_dump(include={"address": {"street", "city"}})
print(address_partial_dict)


# exclude unset fields during serialization:
# as we have set married field with default value False and we have not provided any value for it during model creation
# so it is considered as unset field
# and exclude_unset = True then married field will be excluded from the serialized output
dict_exclude_unset = patient1.model_dump(exclude_unset=True)
print(dict_exclude_unset)


# Deserialization: Create Pydantic model from dictionary
patient_dict = {
    "name": "Bob",
    "gender": "male",
    "age": 30,
    "address": {
        "street": "house no 5, sec22",
        "city": "delhi",
        "state": "delhi",
        "pincode": 110022
    }
}

patient2 = Patient.model_validate(patient_dict)
print(patient2)

{'name': 'Alice', 'gender': 'male', 'age': 25, 'address': {'street': 'house no 2, sec66', 'city': 'gurgaon', 'state': 'haryana', 'pincode': 122002}, 'married': False}
<class 'dict'>
{"name":"Alice","gender":"male","age":25,"address":{"street":"house no 2, sec66","city":"gurgaon","state":"haryana","pincode":122002},"married":false}
<class 'str'>
{'name': 'Alice', 'age': 25}
{"name":"Alice","gender":"male","address":{"street":"house no 2, sec66","city":"gurgaon","state":"haryana","pincode":122002},"married":false}
{'address': {'street': 'house no 2, sec66', 'city': 'gurgaon'}}
{'name': 'Alice', 'gender': 'male', 'age': 25, 'address': {'street': 'house no 2, sec66', 'city': 'gurgaon', 'state': 'haryana', 'pincode': 122002}}
name='Bob' gender='male' age=30 address=Address(street='house no 5, sec22', city='delhi', state='delhi', pincode=110022) married=False
